# Tokyo Olympics 2021 Data Analysis

Analysis of the 2021 Tokyo Summer Olympics dataset containing details of over 11,000 athletes across 47 disciplines and 743 teams. Covers athlete demographics, medal tallies, gender distribution, and coaching statistics.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline

In this task we work with 2021 Tokyo summer Olympics dataset. This contains the details of over 11,000 athletes, with 47 disciplines, along with 743 Teams taking part in the 2021(2020) Tokyo Olympics. This dataset contains the details of the Athletes, Coaches, Teams participating as well as the Entries by gender. It contains their names, countries represented, discipline, gender of competitors, name of the coaches.

In [ ]:
athlete_data = pd.read_excel("Dataset/Athletes.xlsx")
coach_data = pd.read_excel("Dataset/Coaches.xlsx")
gender_data = pd.read_excel("Dataset/EntriesGender.xlsx")
medal_data = pd.read_excel("Dataset/Medals.xlsx")
team_data = pd.read_excel("Dataset/Teams.xlsx")

#### Let's look at the first few rows for each table

In [ ]:
athlete_data.head()

In [ ]:
coach_data.head()

In [ ]:
gender_data.head()

In [ ]:
medal_data.head()

In [ ]:
team_data.head()

### 1. Which countries do most athletes come from? Plot a descendingly ordered bar plot to show athletes counts based on their country of origin? 

In [ ]:
athlete_country= athlete_data["NOC"].value_counts()[:50]
athlete_country

In [ ]:
fig, ax = plt.subplots()
athlete_country.plot(figsize=(10, 5), ax = ax, kind = 'bar', ylabel = 'frequency')
plt.show()

### 2. Which country has the most female athletes? Plot a descendingly ordered bar plot to show female athletes counts based on their country of origin?

In [ ]:
country_women = team_data[team_data["Event"] == "Women"]["NOC"].value_counts()
country_women

In [ ]:
fig, ax = plt.subplots()
country_women.plot(figsize=(10, 5), ax = ax, kind = 'bar', ylabel = 'frequency')
plt.show()

### 3. Which country has the least female athletes?

In [ ]:
all_countries = set(athlete_data["NOC"].unique())

In [ ]:
countries_with_women = set(team_data[team_data["Event"]=="Women"]["NOC"].unique())

In [ ]:
# Countries with no women athletes
countries_without_women = all_countries - countries_with_women
np.array(countries_without_women)

In [ ]:
# finding countries with women participants with least number
countries_with_women = team_data[team_data["Event"]=="Women"]["NOC"].value_counts(ascending=True)

In [ ]:
participation_count = countries_with_women[0]

In [ ]:
countries_with_women[countries_with_women == participation_count].index

### 4. Which sport is most popular (based on athletes counts)  and which country has the highest participants in it? Plot a descendingly ordered bar plot to show athlete counts in different sports? 

In [ ]:
# Most popular sport
most_popular_sport = athlete_data["Discipline"].mode()[0]
most_popular_sport

In [ ]:
# country with highest number of participants in Athletics
country_highest_most_popular_sport = athlete_data[athlete_data["Discipline"] == most_popular_sport]["NOC"].mode()[0]
country_highest_most_popular_sport

In [ ]:
plt.figure(figsize=(15,5))
plot = athlete_data["Discipline"].value_counts().plot(kind='bar')
plt.show()

### 5. Plot a descendingly ordered categorical bar plot to show gender segregated athlete counts in different sports.

In [ ]:
df1 = gender_data.sort_values("Total", ascending=False)[["Discipline", "Male", "Female"]]
plot = df1.plot.bar(figsize=(15,5))
plot.set_xticklabels(df1["Discipline"])
plt.show()

### 6. Which sport has they highest proportion of male to female athletes? Plot a descendingly ordered bar plot to depict male to female athletes proportion across different sports.

In [ ]:
gender_data.head()

In [ ]:
#finding the proportion of male to female athletes
gender_data["male_to_female"] = gender_data.apply(lambda row: row["Male"]/row["Female"], axis=1)
sorted_gender_data = gender_data.sort_values("male_to_female", ascending=False)
sorted_gender_data.head(1)

In [ ]:
# plotting the descendingly ordered bar plot
plot = sorted_gender_data["male_to_female"].plot.bar(figsize=(15,5))
plot.set_xticklabels(sorted_gender_data["Discipline"])
plt.show()

### 7. Which country recieved most gold medals? Which recieved most silver and most bronze? Which received least for each? Use bar plot to for illustration.  

In [ ]:
medal_tally = []
for category in ["Gold", "Silver", "Bronze"]:
    #finding the country with highest number of medals in each category and appending the count to list
    high = medal_data.iloc[medal_data[category].idxmax()][["Team/NOC", category]].values
    medal_tally.append([category+"-High", high[0], high[1]])
    
    #finding the country with lowest number of medals in each category and appending the count to list
    low = medal_data.iloc[medal_data[category].idxmin()][["Team/NOC", category]].values
    medal_tally.append([category+"-Low", low[0], low[1]])
    
#creating a dataframe
medal_tally = pd.DataFrame(medal_tally, columns=["Category", "Country", "Medals"])

In [ ]:
medal_tally

In [ ]:
#plotting the above results
plt.rcParams["figure.figsize"] = (20,10)
ax = medal_tally.plot(kind="bar")
ax.set_title("Medals - Highest and Lowest")
ax.set_ylabel("Medal Count")
ax.set_xticklabels(medal_tally["Category"])

#adding country name to each bar
labels = medal_tally["Country"]

for i, label in enumerate(labels):
    plt.annotate(label, xy=(i, ax.patches[i].get_height()), ha='center', va='bottom')

plt.show()

### 8. Which country has the highest number of medals (Gold – Silver- Bronze) per capita (participants)? 

In [ ]:
# participant count per country
country_participants = athlete_data["NOC"].value_counts().to_frame().reset_index().rename(columns={"index": "Country", "NOC": "ParticipantCount"})

In [ ]:
country_participants.head()

In [ ]:
# Merging the medal_data and the identified country_participants from athlete_data
medal_participants = pd.merge(medal_data, country_participants, left_on='Team/NOC', right_on='Country', how='left').drop("Country", axis=1)

In [ ]:
medal_participants.head()

In [ ]:
# finding per capita participants
for cat in ["Gold", "Silver", "Bronze"]:
    medal_participants[cat+"_percap"] = medal_participants.apply(lambda row: row[cat]/row["ParticipantCount"], axis=1)

In [ ]:
#country with highest number of medals (in each category per capita (participants)
for cat in ["Gold", "Silver", "Bronze"]:
    print(cat, " : ", medal_participants.iloc[medal_participants[cat+"_percap"].idxmax()]["Team/NOC"])

### 9. List countries with no medals? 

In [ ]:
countries_with_medals = set(medal_data["Team/NOC"].unique())

In [ ]:
countries_without_medals = np.array(all_countries - countries_with_medals)

In [ ]:
countries_without_medals

### 10. Which country had the greatest number of coaches. Plot a descendingly ordered bar plot to show coach counts in different countries. 

In [ ]:
plot = coach_data["NOC"].value_counts().plot(kind="bar")
plt.show()

### 11. Plot a descendingly ordered bar plot to show coach counts across different sports.  

In [ ]:
plt.rcParams["figure.figsize"] = (10,5)
plot = coach_data["Discipline"].value_counts().plot(kind="bar")
plt.show()

### 12. Identify the most popular coach, female and male athlete first name? 

In [ ]:
# splitting the name and keeping only the first name
def get_firstname(name):
    return name.split(" ")[0]

In [ ]:
#creating new column with first names of coach and athletes
coach_data["first_name"] = coach_data["Name"].apply(get_firstname)
athlete_data["first_name"] = athlete_data["Name"].apply(get_firstname)

In [ ]:
athlete_data.head()

In [ ]:
coach_data.head()

In [ ]:
# printing the most common coach and athlete with their first names
print("Popular Name:")
print("Coach: ", coach_data["first_name"].mode()[0])
print("Female and Male: ", athlete_data["first_name"].mode()[0])